<a href="https://colab.research.google.com/github/Azmalkhan0/KYC-Esclation-automation-pipeline-module-python/blob/main/KYC_Automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Split the coordinats from url

In [ ]:
import re
import requests
import pandas as pd
import geopandas as gpd

# ==========================================
# 1. INPUT GOOGLE MAPS URL
# ==========================================
url = input("Paste Google Maps URL: ").strip()

# ==========================================
# 2. RESOLVE URL AND EXTRACT COORDINATES
# ==========================================
try:
    response = requests.get(
        url,
        allow_redirects=True,
        timeout=25,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    final_url = response.url
    print("\nResolved URL:", final_url)

except requests.RequestException as e:
    raise RuntimeError(f"Could not open Google Maps URL: {e}")

patterns = [
    r'@(-?\d+(?:\.\d+)?),(-?\d+(?:\.\d+)?)',
    r'[?&]q=(-?\d+(?:\.\d+)?),(-?\d+(?:\.\d+)?)',
    r'!3d(-?\d+(?:\.\d+)?)!4d(-?\d+(?:\.\d+)?)'
]

coordinates = None

for pattern in patterns:
    match = re.search(pattern, final_url)

    if match:
        coordinates = (
            float(match.group(1)),
            float(match.group(2))
        )
        break

if coordinates is None:
    raise ValueError("Latitude and longitude not found in the URL.")

latitude, longitude = coordinates

if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
    raise ValueError("Invalid latitude or longitude.")

print("\nLatitude :", latitude)
print("Longitude:", longitude)

# ==========================================
# 3. LOAD PINCODE GEOJSON
# ==========================================
geojson_path = (
    "/content/All_India_pincode_Boundary-19312 (1).geojson"
)

gdf = gpd.read_file(geojson_path)

if gdf.crs is None:
    raise ValueError("GeoJSON CRS is missing. Verify its coordinate system.")

gdf = gdf.to_crs("EPSG:4326")

pin_col = next(
    (col for col in gdf.columns if col.lower() == "pincode"),
    None
)

if pin_col is None:
    raise ValueError(f"Pincode column not found: {list(gdf.columns)}")

gdf = gdf.rename(columns={pin_col: "predicted_api_pincode"})

gdf = gdf[
    gdf.geometry.notna() & ~gdf.geometry.is_empty
].copy()

# ==========================================
# 4. CREATE GEOCODE POINT
# ==========================================
point = gpd.GeoDataFrame(
    {
        "latitude": [latitude],
        "longitude": [longitude]
    },
    geometry=gpd.points_from_xy([longitude], [latitude]),
    crs="EPSG:4326"
)

# ==========================================
# 5. FIND PINCODE
# ==========================================
result = gpd.sjoin(
    point,
    gdf[["predicted_api_pincode", "geometry"]],
    how="left",
    predicate="within"
)

pincodes = result["predicted_api_pincode"].dropna().unique()

# ==========================================
# 6. DISPLAY FINAL OUTPUT
# ==========================================
output = pd.DataFrame([{
    "Google_Maps_URL": url,
    "latitude": latitude,
    "longitude": longitude,
    "predicted_api_pincode": (
        ", ".join(map(str, pincodes))
        if len(pincodes) else None
    ),
    "geocode_remark": "Matched" if len(pincodes) else "Not Found"
}])

print("\n========== FINAL OUTPUT ==========")
print(output.to_string(index=False))

# Selenium depandacies and chrome webdriver

In [ ]:
!apt-get update -qq
!apt-get install -y -qq wget gnupg

!wget -q https://dl.google.com/linux/linux_signing_key.pub \
    -O /tmp/google-linux-signing-key.pub

!gpg --dearmor \
    -o /usr/share/keyrings/google-chrome.gpg \
    /tmp/google-linux-signing-key.pub

!echo "deb [arch=amd64 signed-by=/usr/share/keyrings/google-chrome.gpg] https://dl.google.com/linux/chrome/deb/ stable main" \
    > /etc/apt/sources.list.d/google-chrome.list

!apt-get update -qq
!apt-get install -y -qq google-chrome-stable
!pip install -q --upgrade selenium

# Geocode Validation

In [ ]:
# ============================================================
# GOOGLE MAPS SEARCH + LANDMARK EXTRACTION + NEAREST GEOCODE
# (landmark first, company-name fallback, no API)
# Google Colab
# ============================================================
#
# Workflow:
#   1. Search using LANDMARK + last address part.
#      - nearest result <= 1 km  -> update
#   2. If that fails AND a landmark was actually used, search
#      using COMPANY NAME + last address part.
#      - nearest result <= 1 km  -> update (basis: company name)
#   3. Otherwise -> not update
#
# Scraping notes (why it now behaves like a normal browser):
#   - Searches are biased to the EXISTING coordinates (@lat,lon)
#   - Browser location / timezone / language are set to India
#   - Consent pages are clicked through automatically
#   - Waits for the results to appear instead of a fixed sleep
#   - Tries 3 different Google Maps URL formats before giving up
#   - Handles the "single match opens the place page directly" case
# ============================================================

import re
import math
import time
import shutil
import pandas as pd

from urllib.parse import quote
from IPython.display import display
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException, WebDriverException


# ============================================================
# 1. USER INPUTS
# ============================================================

company_name = input("Enter Company Name: ").strip()
full_address = input("Enter Full Address: ").strip()

existing_latitude = float(input("Enter Existing Latitude: ").strip())
existing_longitude = float(input("Enter Existing Longitude: ").strip())

DISTANCE_THRESHOLD_KM = 1.0
MAX_PLACES_TO_VISIT = 10
DEBUG_SCREENSHOT = "/content/debug_no_results.png"


if not (
    math.isfinite(existing_latitude)
    and math.isfinite(existing_longitude)
    and -90 <= existing_latitude <= 90
    and -180 <= existing_longitude <= 180
):
    raise ValueError("Please enter valid existing latitude and longitude.")


# ============================================================
# 2. LANDMARK TRIGGERS
# ============================================================

LANDMARK_TRIGGERS = [
    "near by",
    "in front of",
    "opposite to",
    "near",
    "nearby",
    "opposite",
    "opp of",
    "opp.",
    "opp",
    "beside",
    "behind",
    "next to",
    "adjacent to",
    "infront of",
    "close to",
    "back side of",
    "backside of",
    "behind of",
    "outside",
    "at the back of",
    "in the front of",
    "nearer to"
]

# Match longer triggers before shorter triggers
LANDMARK_TRIGGERS = sorted(LANDMARK_TRIGGERS, key=len, reverse=True)

trigger_pattern = re.compile(
    r"(?<!\w)("
    + "|".join(re.escape(t) for t in LANDMARK_TRIGGERS)
    + r")(?!\w)",
    flags=re.IGNORECASE
)


# ============================================================
# 3. EXTRACT LANDMARK / SEARCH NAME
# ============================================================

def extract_landmark(company, address):
    """
    Search comma-separated address segments from right to left.
    Extract the text after the last applicable landmark trigger.

    Returns (name, landmark_found)
      - landmark_found True  -> name is the landmark text
      - landmark_found False -> name is the company name
    """

    segments = [s.strip() for s in address.split(",") if s.strip()]

    for segment in reversed(segments):
        matches = list(trigger_pattern.finditer(segment))

        if not matches:
            continue

        match = matches[-1]
        landmark = segment[match.end():].strip()

        landmark = re.sub(r"^[\s:;.\-]+", "", landmark)
        landmark = re.sub(r"[\s;]+$", "", landmark)

        if landmark:
            return landmark, True

    return company.strip(), False


# ============================================================
# 4. EXTRACT FINAL ADDRESS COMPONENT
# ============================================================

def extract_address(address):
    """Use the last non-empty comma-separated component."""

    segments = [s.strip() for s in address.split(",") if s.strip()]
    return segments[-1] if segments else ""


extracted_landmark, landmark_found = extract_landmark(
    company_name, full_address
)
extracted_address = extract_address(full_address)

# Query 1: landmark based (or company based if no landmark exists)
landmark_query = ", ".join(
    p for p in [extracted_landmark, extracted_address] if p
)

# Query 2: company-name based (fallback)
company_query = ", ".join(
    p for p in [company_name, extracted_address] if p
)

print("\n========== INPUT DETAILS ==========")
print("Company Name        :", company_name)
print("Full Address        :", full_address)
print("Landmark Found      :", landmark_found)
print("Extracted Name      :", extracted_landmark)
print("Extracted Address   :", extracted_address)
print("Landmark Query      :", landmark_query)
print("Company Query       :", company_query)
print("Existing Latitude   :", existing_latitude)
print("Existing Longitude  :", existing_longitude)


# ============================================================
# 5. DISTANCE CALCULATION
# ============================================================

def haversine_distance(lat1, lon1, lat2, lon2):
    """Straight-line distance between two coordinates, in km."""

    earth_radius_km = 6371.0

    lat1, lon1, lat2, lon2 = map(
        math.radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    a = max(0.0, min(1.0, a))

    return 2 * earth_radius_km * math.asin(math.sqrt(a))


# ============================================================
# 6. EXTRACT COORDINATES FROM GOOGLE MAPS URL
# ============================================================

def extract_coordinates_from_url(url):
    """
    Prefer place coordinates (!3dLAT!4dLON).
    Fallback to the map viewport (@LAT,LON), which may be
    approximate.
    """

    if not url:
        return None, None

    match = re.search(
        r"!3d(-?\d+(?:\.\d+)?)!4d(-?\d+(?:\.\d+)?)", url
    )
    if match:
        return float(match.group(1)), float(match.group(2))

    match = re.search(
        r"/@(-?\d+(?:\.\d+)?),(-?\d+(?:\.\d+)?)(?:,|/)", url
    )
    if match:
        return float(match.group(1)), float(match.group(2))

    return None, None


# ============================================================
# 7. CHROME / SELENIUM SETUP
# ============================================================

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("--lang=en-IN")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])

chrome_path = (
    shutil.which("google-chrome")
    or shutil.which("google-chrome-stable")
    or shutil.which("chromium")
    or shutil.which("chromium-browser")
)

if chrome_path:
    options.binary_location = chrome_path

driver_path = (
    shutil.which("chromedriver")
    or shutil.which("chromium-chromedriver")
)

driver = None


def start_driver():
    """
    Start Chrome and make it look like a normal browser used from
    India: matching user agent, India language/timezone, and the
    existing coordinates as the browser location.
    """

    if driver_path:
        d = webdriver.Chrome(
            service=Service(driver_path), options=options
        )
    else:
        d = webdriver.Chrome(options=options)

    d.set_page_load_timeout(45)

    try:
        version = d.capabilities.get("browserVersion", "120.0.0.0")
        user_agent = (
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
            f"(KHTML, like Gecko) Chrome/{version} Safari/537.36"
        )

        d.execute_cdp_cmd(
            "Network.setUserAgentOverride",
            {
                "userAgent": user_agent,
                "acceptLanguage": "en-IN,en;q=0.9"
            }
        )
    except Exception:
        pass

    for command, params in [
        (
            "Emulation.setGeolocationOverride",
            {
                "latitude": existing_latitude,
                "longitude": existing_longitude,
                "accuracy": 100
            }
        ),
        ("Emulation.setTimezoneOverride", {"timezoneId": "Asia/Kolkata"}),
        ("Emulation.setLocaleOverride", {"locale": "en-IN"}),
    ]:
        try:
            d.execute_cdp_cmd(command, params)
        except Exception:
            pass

    return d


# ============================================================
# 8. GOOGLE MAPS HELPERS
# ============================================================

def build_search_urls(query):
    """
    Several Google Maps URL formats, tried in order.
    The first one is biased to the existing coordinates, which
    mimics searching from the right area in a normal browser.
    """

    # "Colony-281121" -> "Colony 281121"
    cleaned = re.sub(r"(?<=[A-Za-z])\s*-\s*(?=\d)", " ", query)
    encoded = quote(cleaned, safe="")

    return [
        (
            f"https://www.google.com/maps/search/{encoded}/"
            f"@{existing_latitude},{existing_longitude},14z"
            "?hl=en&gl=in"
        ),
        f"https://www.google.com/maps/search/{encoded}?hl=en&gl=in",
        (
            "https://www.google.com/maps/search/?api=1&query="
            f"{encoded}&hl=en&gl=in"
        ),
    ]


def handle_consent(d):
    """Click through a Google consent page if one is shown."""

    try:
        body_text = d.find_element(By.TAG_NAME, "body").text.lower()
    except Exception:
        body_text = ""

    if (
        "consent.google" not in d.current_url
        and "before you continue" not in body_text
    ):
        return False

    for label in ["Accept all", "Reject all", "I agree"]:
        try:
            buttons = d.find_elements(
                By.XPATH, f'//button[contains(., "{label}")]'
            )
            for button in buttons:
                try:
                    button.click()
                    print(f"  Consent page: clicked '{label}'")
                    time.sleep(3)
                    return True
                except Exception:
                    continue
        except Exception:
            continue

    return False


def wait_for_results_or_place(d, timeout=25):
    """
    Wait until Google Maps shows either a results list or a single
    place page, instead of sleeping for a fixed time.
    """

    def ready(driver_):
        if driver_.find_elements(By.CSS_SELECTOR, 'div[role="feed"]'):
            return True
        if driver_.find_elements(
            By.CSS_SELECTOR, 'a[href*="/maps/place/"]'
        ):
            return True
        if (
            "/maps/place/" in driver_.current_url
            and driver_.find_elements(By.TAG_NAME, "h1")
        ):
            return True
        return False

    try:
        WebDriverWait(d, timeout).until(ready)
        return True
    except TimeoutException:
        return False


def collect_place_urls(d):
    """Scroll the results list and collect unique place URLs."""

    place_urls = []

    # Scroll the results panel to load more results
    for _ in range(4):
        try:
            d.execute_script("""
                const panels = [
                    ...document.querySelectorAll('div[role="feed"]')
                ];
                if (panels.length) {
                    panels[0].scrollTop = panels[0].scrollHeight;
                }
            """)
            time.sleep(1.5)
        except WebDriverException:
            break

    for link in d.find_elements(
        By.CSS_SELECTOR, 'a[href*="/maps/place/"]'
    ):
        try:
            href = link.get_attribute("href")
            if (
                href
                and "/maps/place/" in href
                and href not in place_urls
            ):
                place_urls.append(href)
        except WebDriverException:
            continue

    # A single match opens the place page directly (no list)
    if not place_urls and "/maps/place/" in d.current_url:
        place_urls.append(d.current_url)

    return place_urls


def print_no_result_diagnostics(d):
    """Explain what the browser actually saw."""

    print("  Page title :", d.title)
    print("  Page URL   :", d.current_url)

    try:
        text = d.find_element(By.TAG_NAME, "body").text.lower()

        if "before you continue" in text or "consent" in d.current_url:
            print("  Note: Google consent page is blocking the search.")
        elif "can't find" in text or "no results" in text:
            print("  Note: Google Maps has no match for this query.")
        elif "unusual traffic" in text or "captcha" in text:
            print("  Note: Google is asking for a CAPTCHA (bot check).")
    except Exception:
        pass

    try:
        d.save_screenshot(DEBUG_SCREENSHOT)
        print("  Screenshot saved:", DEBUG_SCREENSHOT)
    except Exception:
        pass


# ============================================================
# 9. GOOGLE MAPS SEARCH AND RESULT COLLECTION
# ============================================================

def collect_google_maps_results(query):
    """
    Search Google Maps and collect available place results
    (name, address, coordinates, URL). Best-effort extraction.
    """

    global driver

    results = []
    place_urls = []

    try:
        driver = start_driver()
        wait = WebDriverWait(driver, 20)

        # Try each URL format until one returns places
        for variant_no, search_url in enumerate(
            build_search_urls(query), 1
        ):
            print(f"  URL format {variant_no}...")

            try:
                driver.get(search_url)
            except TimeoutException:
                print("  Page load timed out; continuing.")
            except WebDriverException as error:
                print("  Could not open page:", error)
                continue

            handle_consent(driver)
            wait_for_results_or_place(driver)
            time.sleep(2)

            place_urls = collect_place_urls(driver)

            if place_urls:
                break

            print_no_result_diagnostics(driver)

        print("Google Maps place URLs found:", len(place_urls))

        # Visit each place and extract its details
        for result_no, place_url in enumerate(
            place_urls[:MAX_PLACES_TO_VISIT], 1
        ):
            try:
                # Skip reloading if we're already on this place page
                if driver.current_url != place_url:
                    driver.get(place_url)

                try:
                    wait.until(
                        lambda d: d.find_elements(By.TAG_NAME, "h1")
                    )
                except TimeoutException:
                    pass

                time.sleep(1.5)

                # Name
                google_name = ""
                try:
                    google_name = driver.find_element(
                        By.CSS_SELECTOR, "h1.DUwDvf"
                    ).text.strip()
                except Exception:
                    try:
                        google_name = driver.find_element(
                            By.TAG_NAME, "h1"
                        ).text.strip()
                    except Exception:
                        google_name = ""

                # Address
                google_address = ""
                address_selectors = [
                    'button[data-item-id="address"]',
                    'button[aria-label^="Address:"]',
                    '[data-item-id="address"]'
                ]

                for selector in address_selectors:
                    try:
                        el = driver.find_element(
                            By.CSS_SELECTOR, selector
                        )
                        google_address = (
                            el.get_attribute("aria-label")
                            or el.text
                            or ""
                        ).strip()

                        google_address = re.sub(
                            r"^Address:\s*", "", google_address,
                            flags=re.IGNORECASE
                        )

                        if google_address:
                            break
                    except Exception:
                        continue

                current_url = driver.current_url

                latitude, longitude = extract_coordinates_from_url(
                    current_url
                )

                if latitude is None or longitude is None:
                    latitude, longitude = extract_coordinates_from_url(
                        place_url
                    )

                results.append({
                    "Result_No": result_no,
                    "Google_Name": google_name,
                    "Google_Address": google_address,
                    "Latitude": latitude,
                    "Longitude": longitude,
                    "Place_URL": current_url or place_url
                })

            except Exception as error:
                print(f"Could not extract result {result_no}: {error}")

                results.append({
                    "Result_No": result_no,
                    "Google_Name": "",
                    "Google_Address": "",
                    "Latitude": None,
                    "Longitude": None,
                    "Place_URL": place_url
                })

    except Exception as error:
        print("\nGoogle Maps / Selenium error:", error)
        print(
            "Check that Chrome/Chromium and a compatible "
            "ChromeDriver are installed."
        )

    finally:
        if driver is not None:
            try:
                driver.quit()
            except Exception:
                pass
            driver = None

    return results


# ============================================================
# 10. SELECT THE NEAREST RESULT FOR ONE SEARCH ATTEMPT
# ============================================================

def select_nearest_result(
    results,
    query,
    basis,
    threshold_km=1.0
):
    """
    Evaluate one search attempt.

    Distance <= threshold: update
    Distance > threshold : not update
    No valid coordinates : not update
    """

    output = {
        "Search_Basis": basis,
        "Search_Query": query,
        "Google_Results_Found": len(results),
        "Google_Name": None,
        "Google_Address": None,
        "Latitude": None,
        "Longitude": None,
        "Distance_km": None,
        "Place_URL": None,
        "Remark": "not update",
        "Status": ""
    }

    if not results:
        output["Status"] = "No Google Maps results found"
        return output

    valid_results = []

    for result in results:
        try:
            lat = float(result["Latitude"])
            lon = float(result["Longitude"])

            if not (
                math.isfinite(lat)
                and math.isfinite(lon)
                and -90 <= lat <= 90
                and -180 <= lon <= 180
            ):
                continue

            candidate = result.copy()
            candidate["Distance_km"] = haversine_distance(
                existing_latitude, existing_longitude, lat, lon
            )
            valid_results.append(candidate)

        except (TypeError, ValueError, KeyError):
            continue

    if not valid_results:
        output["Status"] = "No results with valid coordinates"
        return output

    nearest = min(valid_results, key=lambda item: item["Distance_km"])
    distance_km = nearest["Distance_km"]
    within = distance_km <= threshold_km

    output.update({
        "Google_Name": nearest.get("Google_Name"),
        "Google_Address": nearest.get("Google_Address"),
        "Latitude": nearest.get("Latitude"),
        "Longitude": nearest.get("Longitude"),
        "Distance_km": round(distance_km, 3),
        "Place_URL": nearest.get("Place_URL"),
        "Remark": "update" if within else "not update",
        "Status": (
            f"Nearest result selected ({basis} search)"
            if within
            else f"Nearest result is more than {threshold_km} km away "
                 f"({basis} search)"
        )
    })

    return output


def add_audit_distances(results, basis, query):
    """Return all collected results with distances, for checking."""

    rows = []

    for result in results:
        row = result.copy()
        row["Search_Basis"] = basis
        row["Search_Query"] = query

        try:
            lat = float(row["Latitude"])
            lon = float(row["Longitude"])

            if (
                math.isfinite(lat)
                and math.isfinite(lon)
                and -90 <= lat <= 90
                and -180 <= lon <= 180
            ):
                row["Distance_km"] = round(
                    haversine_distance(
                        existing_latitude, existing_longitude, lat, lon
                    ),
                    3
                )
            else:
                row["Distance_km"] = None

        except (TypeError, ValueError, KeyError):
            row["Distance_km"] = None

        rows.append(row)

    return rows


# ============================================================
# 11. RUN THE COMPLETE WORKFLOW (LANDMARK -> COMPANY FALLBACK)
# ============================================================

audit_rows = []

# ---- Attempt 1: landmark based (company based if no landmark) ----
first_basis = "Landmark" if landmark_found else "Company name"

print(f"\n>>> Attempt 1 [{first_basis}]: {landmark_query}")
results_1 = collect_google_maps_results(landmark_query)
attempt_1 = select_nearest_result(
    results_1, landmark_query, first_basis, DISTANCE_THRESHOLD_KM
)
audit_rows += add_audit_distances(results_1, first_basis, landmark_query)

print(
    f"Attempt 1 result: {attempt_1['Remark']} "
    f"(distance: {attempt_1['Distance_km']} km)"
)

attempt_2 = None
final_attempt = attempt_1

# ---- Attempt 2: company-name fallback ----
# Only when attempt 1 failed AND it was really landmark based
# (otherwise the company query would just repeat attempt 1).
if attempt_1["Remark"] != "update" and landmark_found:

    print(f"\n>>> Attempt 2 [Company name]: {company_query}")
    results_2 = collect_google_maps_results(company_query)
    attempt_2 = select_nearest_result(
        results_2, company_query, "Company name", DISTANCE_THRESHOLD_KM
    )
    audit_rows += add_audit_distances(
        results_2, "Company name", company_query
    )

    print(
        f"Attempt 2 result: {attempt_2['Remark']} "
        f"(distance: {attempt_2['Distance_km']} km)"
    )

    if attempt_2["Remark"] == "update":
        final_attempt = attempt_2
    else:
        # Both failed: report whichever got closest, as not update
        candidates = [
            a for a in (attempt_1, attempt_2)
            if a["Distance_km"] is not None
        ]
        if candidates:
            final_attempt = min(candidates, key=lambda a: a["Distance_km"])
        else:
            final_attempt = attempt_1


# ---- Build final output row ----
final_result = {
    "Input_Company_Name": company_name,
    "Input_Full_Address": full_address,
    "Extracted_Name": extracted_landmark,
    "Extracted_Address": extracted_address,
    "Existing_Latitude": existing_latitude,
    "Existing_Longitude": existing_longitude,
    "Landmark_Query": landmark_query if landmark_found else None,
    "Landmark_Distance_km": (
        attempt_1["Distance_km"] if landmark_found else None
    ),
    "Company_Query": company_query,
    "Company_Distance_km": (
        attempt_2["Distance_km"] if attempt_2
        else (attempt_1["Distance_km"] if not landmark_found else None)
    ),
}

final_result.update(final_attempt)

if final_attempt["Remark"] != "update" and attempt_2 is not None:
    final_result["Status"] = (
        f"Neither landmark nor company-name search found a result "
        f"within {DISTANCE_THRESHOLD_KM} km"
    )

# ---- Output as DataFrames (no files are saved) ----
# final_df : one row, the final decision (update / not update)
# audit_df : every Google Maps result collected, with distances
final_df = pd.DataFrame([final_result])
audit_df = pd.DataFrame(audit_rows)

print("\n========== FINAL RESULT (final_df) ==========")
display(final_df)

print("\n========== ALL COLLECTED RESULTS (audit_df) ==========")
display(audit_df)